# 02 — A 240 KiB-anchored two-line model and a separately learned breakpoint

**Run All reuses the saved CSVs by default.** Set `RERUN_MEASUREMENTS = True` only to perform another native SSD sweep. A new sweep is stored in a new sibling directory; it never replaces the previous measurements. Missing, incomplete, or incompatible cached data produces an error, not an implicit benchmark.

We compare four breakpoint choices in the **same constrained model**: fixed **240**, **256**, **230** KiB and a separately trained $s_T$. The unrestricted hinge and affine models remain baselines. The primary fixed model is **$T_{240}$**, not the $s_{99}$-anchored fit.

The three reference values came from different operations in the previous seven-block run: raw all-block $T(x)/x$ argmin (240), smoothed 99% throughput crossing (256), and a fitted free hinge (230). They are not interchangeable definitions of a physical saturation length. In particular, 240 was noticed using all seven blocks: its fixed comparison is **retrospective**, not an independent new holdout experiment.

## Logical request size and the constrained $T_s$ family

$x$ is logical chunk length in **KiB**, not rows; $8192x$ is its bit length. From block $b$'s estimated multi-chunk throughput,

$$\widehat T_b(x)=\frac{1000x}{1024\widehat B_b(x)}\ \mathrm{ms/chunk}.$$

This is amortized, throughput-derived latency, **not the wall time of one isolated read**. The profiler's `saturation_throughput` is an estimator, not proof of a sustained plateau.

For positive requests, fit

$$T_s(x)=b_1x+d\max(s,x)=\begin{cases}ds+b_1x,&0<x\le s,\\(b_1+d)x,&x\ge s,\end{cases}\qquad b_1,d\ge0.$$

Thus the intercept is constrained to $ds$ and $T_s(x)/x$ is constant after $s$. This differs from the free hinge $a+bx+q(x-h)_+$, whose tail can retain a nonzero intercept. A zero-size request has zero cost and is outside the fitted positive-size grid.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for root in roots for p in (root, root / "conclusion/python")
             if (p / "chunk_latency.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from this repository or conclusion/python.")
sys.path.insert(0, str(ROOT))
import importlib
import chunk_latency
chunk_latency = importlib.reload(chunk_latency)

# The only switch that can enable native I/O. No torch/submodule is needed otherwise.
RERUN_MEASUREMENTS = False
DATA_DIR = ROOT / "runs" / "02_chunk_latency"
SEED, TRAIN_BLOCKS, BOOTSTRAP_REPEATS = 0, 4, 1000
MEASUREMENT_CONFIG = dict(chunk_latency.DEFAULT_CONFIG, seed=SEED)
# Original protocol: 384 sizes, 7 blocks, 10 samples, 3 warmups,
# 6 reader threads, 128 MiB blob. Changing it invalidates an incompatible cache.

In [ ]:
RUN, raw, estimates, hardware = chunk_latency.acquire(
    DATA_DIR, rerun=RERUN_MEASUREMENTS,
    submodule=ROOT.parents[1] / "preliminary_research/vlm-flash",
    config=MEASUREMENT_CONFIG,
)
# Do NOT call reporting.begin(): it deletes the run, including the cached CSVs.
OUT = RUN / "breakpoint_comparison"
OUT.mkdir(parents=True, exist_ok=True)
source_hashes = chunk_latency.input_hashes(RUN)
provenance = dict(source_directory=str(RUN), source_sha256=source_hashes,
    source_hardware=hardware, remeasured=RERUN_MEASUREMENTS,
    train_blocks=list(range(TRAIN_BLOCKS)),
    holdout_blocks=list(range(TRAIN_BLOCKS, hardware["blocks"])),
    selection_objective="unweighted training MSE on the measured positive-size grid",
    primary_model="Ts@240", fixed_breakpoints_kib=[240, 256, 230])
(OUT / "input_provenance.json").write_text(json.dumps(provenance, indent=2))
display(Markdown(f"**{'New measurements' if RERUN_MEASUREMENTS else 'Reused measurements'}:** `{RUN}`\n\n"
                 f"**Analysis output:** `{OUT}`. Native SSD sweep: **{RERUN_MEASUREMENTS}**."))
print(f"{len(estimates):,} block/size estimates; {len(raw):,} raw timings")

def figure(name):
    plt.tight_layout()
    plt.savefig(OUT / f"{name}.png", dpi=140, bbox_inches="tight")
    plt.savefig(OUT / f"{name}.svg", bbox_inches="tight")
    plt.show()
    plt.close()

## Learn on four blocks, then evaluate frozen models on three blocks

The first four block medians train coefficients and both free breakpoint searches. The last three block medians are never passed to `train_models`. Every fixed $s$ gets its own nonnegative least-squares coefficient fit. The learned $s_T$ minimizes training MSE over the same interior measured grid used for the free hinge (`x[3:-3]`). This is a finite-grid optimum, not an assertion about the exact continuous or physical breakpoint.

The comparison retains all four $T_s$ rows even when two fitted breakpoints coincide. The measured all-block efficiency argmin and $s_{99}$ are recomputed below rather than silently relabeled as a trained $T_s$ breakpoint.

In [ ]:
curve, samples = chunk_latency.split_curve(estimates, TRAIN_BLOCKS)
x = curve.size_kib.to_numpy()
y, train, holdout = (curve[c].to_numpy() for c in ("median_ms", "train_ms", "holdout_ms"))
curve.to_csv(OUT / "latency_curve.csv", index=False)
parameters, search = chunk_latency.train_models(x, train)
fits, predictions = chunk_latency.compare_models(x, holdout, parameters)
order = ["Ts@240", "Ts@256", "Ts@230", "Ts learned", "Hinge", "Affine"]
fits = fits.set_index("model").loc[order].reset_index()
fits.to_csv(OUT / "model_comparison.csv", index=False)
search.to_csv(OUT / "breakpoint_search.csv", index=False)
(OUT / "model_parameters.json").write_text(json.dumps(parameters, indent=2))
prediction_table = pd.DataFrame(dict(size_kib=x, train_ms=train, holdout_ms=holdout, **predictions))
prediction_table.to_csv(OUT / "model_predictions.csv", index=False)
display(fits)

s99, train_throughput, smooth, peak = chunk_latency.throughput_threshold(x, train)
s99_all, all_throughput, all_smooth, _ = chunk_latency.throughput_threshold(x, y)
s_argmin = float(x[np.argmin(y / x)])
s_near = float(x[np.flatnonzero(y / x <= 1.05 * (y / x).min())[0]])
s_hinge = parameters["Hinge"]["s_kib"]
s_ts = parameters["Ts learned"]["s_kib"]
lengths = pd.DataFrame([
    ("99% smoothed throughput crossing", s99, "first 4 blocks; threshold rule"),
    ("Raw efficiency argmin", s_argmin, "all 7 blocks; descriptive, not independent holdout"),
    ("Free hinge breakpoint", s_hinge, "first 4 blocks; free-hinge training MSE"),
    ("Constrained Ts breakpoint", s_ts, "first 4 blocks; constrained-Ts training MSE"),
], columns=["quantity", "s_kib", "selection_data_and_objective"])
lengths.to_csv(OUT / "breakpoint_definitions.csv", index=False)
display(lengths)
print(f"Primary fixed model: Ts@240. Independently trained Ts breakpoint: {s_ts:g} KiB.")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(x, holdout * 1000, ".", markersize=3, label="Holdout measurements")
for name in order[:-1]:
    plt.plot(x, predictions[name] * 1000, label=f"{name} (s={parameters[name]['s_kib']:g})")
plt.xlabel("Logical chunk length (KiB)")
plt.ylabel("Amortized latency (microseconds/chunk)")
plt.legend()
figure("latency_models_holdout")

plt.figure(figsize=(10, 4))
for name in order[:-1]:
    plt.plot(x, 1000 * (predictions[name] - holdout), label=name)
plt.axhline(0, linestyle=":")
plt.xlabel("Logical chunk length (KiB)")
plt.ylabel("Prediction minus holdout (microseconds/chunk)")
plt.legend()
figure("holdout_residuals")

plt.figure(figsize=(10, 4))
for kind, group in search.groupby("kind"):
    plt.plot(group.s_kib, group.train_rmse_us, label=f"{kind}: train only")
for label, point in [("fixed 240", 240), ("fixed 256", 256), ("fixed 230", 230), ("learned Ts", s_ts)]:
    plt.axvline(point, linestyle="--", linewidth=1, label=f"{label}: {point:g} KiB")
plt.xlabel("Candidate breakpoint (KiB)")
plt.ylabel("Training RMSE (microseconds/chunk)")
plt.legend()
figure("breakpoint_training_objective")

## A first crossing is not a sustained plateau

The 99% rule uses a three-point median of throughput. The efficiency argmin uses the unsmoothed all-block median of latency divided by size. The $T_s$ fit instead minimizes latency error. Even a good $T_s$ fit does not establish that every measured length after the fitted boundary has identical $T(x)/x$.

In [ ]:
pd.DataFrame(dict(size_kib=x, train_mib_s=train_throughput, train_smoothed_mib_s=smooth,
    all_blocks_mib_s=all_throughput, all_blocks_smoothed_mib_s=all_smooth)).to_csv(
    OUT / "throughput_by_length.csv", index=False)
plt.figure(figsize=(10, 4))
plt.plot(x, train_throughput, alpha=.4, label="Train throughput")
plt.plot(x, smooth, label="Train 3-point median")
plt.axhline(.99 * peak, linestyle=":", label="99% of observed smoothed peak")
plt.axvline(s99, linestyle="--", label=f"First crossing: {s99:g} KiB")
plt.xlabel("Logical chunk length (KiB)")
plt.ylabel("Throughput (MiB/s)")
plt.legend()
figure("throughput_threshold")

plt.figure(figsize=(10, 4))
plt.plot(x, 1000 * y / x, ".-", markersize=2, label="All-block median / size")
plt.plot(x, 1000 * holdout / x, alpha=.5, label="Holdout / size")
for name in order[:4]:
    plt.plot(x, 1000 * predictions[name] / x, label=name)
plt.ylim(0, 2 * 1000 * (y / x).min())
plt.xlabel("Logical chunk length (KiB)")
plt.ylabel("Latency / size (microseconds/KiB)")
plt.legend()
figure("efficiency_and_breakpoints")

## Repeated-block uncertainty and finite-count diagnostics

Bootstrap all seven acquisition blocks together across all lengths to describe the original first-crossing/argmin rules. These resamples are descriptive and **are not used to train $s_T$** or tune a model on holdout. Their quantiles are not confidence intervals for a unique physical saturation point. The separate training-objective curve above shows the constrained-model breakpoint sensitivity.

In [ ]:
boot_rng = np.random.default_rng(SEED + 1)
bootstrap = []
for _ in range(BOOTSTRAP_REPEATS):
    sampled = samples[:, boot_rng.integers(0, samples.shape[1], samples.shape[1])]
    median_ms = np.median(sampled, axis=1)
    rho = median_ms / x
    bootstrap.append((chunk_latency.throughput_threshold(x, median_ms)[0],
        x[rho.argmin()], x[np.flatnonzero(rho <= 1.05 * rho.min())[0]]))
boot = pd.DataFrame(bootstrap, columns=["saturation_99pct_kib", "argmin_kib", "near_best_kib"])
boot.to_csv(OUT / "bootstrap_selections.csv", index=False)
display(boot.quantile([.05, .95]))
plt.figure(figsize=(10, 4))
for column in boot:
    counts = boot[column].value_counts().sort_index()
    plt.plot(counts.index, counts.to_numpy() / len(boot), ".-", label=column)
plt.xlabel("Selected length (KiB)")
plt.ylabel("Bootstrap selection frequency")
plt.legend()
figure("bootstrap_selection")

count_curve = raw.groupby(["block", "size_kib", "count"], as_index=False).io_us.median()
count_curve["total_io_mib"] = count_curve.size_kib * count_curve["count"] / 1024
count_curve["throughput_mib_s"] = count_curve.total_io_mib / (count_curve.io_us / 1e6)
count_curve.to_csv(OUT / "count_throughput.csv", index=False)
changes = []
for (block, size), group in count_curve.groupby(["block", "size_kib"]):
    last = group.sort_values("count").tail(2)
    if len(last) == 2:
        a, b = last.throughput_mib_s.to_numpy()
        changes.append(dict(block=int(block), size_kib=float(size), tail_count_change_pct=100 * (b / a - 1)))
tail_changes = pd.DataFrame(changes)
tail_changes.to_csv(OUT / "tail_count_change.csv", index=False)
plt.figure(figsize=(10, 4))
for size in (16, 64, 128, 240, 256, 512, 768):
    group = count_curve[count_curve.size_kib == size].groupby("total_io_mib").throughput_mib_s.median()
    if len(group):
        plt.plot(group.index, group.to_numpy(), ".-", label=f"{size} KiB")
plt.xscale("log")
plt.xlabel("Total logical I/O (MiB)")
plt.ylabel("Median throughput across blocks (MiB/s)")
plt.legend()
figure("count_convergence")

In [ ]:
tail = x >= s99
holdout_ratio = holdout / x
summary = dict(
    primary_model="Ts@240", fixed_breakpoints_kib=[240, 256, 230],
    saturation_99pct_kib=s99, all_blocks_saturation_99pct_kib=s99_all,
    argmin_kib=s_argmin, near_best_5pct_kib=s_near,
    hinge_breakpoint_kib=s_hinge, ts_learned_breakpoint_kib=s_ts,
    ts_search_grid_kib=search[search.kind == "Ts"].s_kib.tolist(),
    ts_breakpoint_identified=parameters["Ts learned"]["breakpoint_identified"],
    ts_breakpoint_at_search_edge=parameters["Ts learned"]["breakpoint_at_search_edge"],
    selection_objective="training MSE only; no holdout breakpoint selection",
    post_s_below_99pct_count=int(np.sum(smooth[tail] < .99 * peak)),
    post_s_count=int(tail.sum()),
    post_s_ratio_cv_pct=float(100 * np.std(holdout_ratio[tail], ddof=1) / np.mean(holdout_ratio[tail])),
    post_s_ratio_endpoint_change_pct=float(100 * (holdout_ratio[tail][-1] / holdout_ratio[tail][0] - 1)),
    minimum_at_upper_boundary=bool(s_argmin == x[-1]),
    throughput_peak_at_upper_boundary=bool(x[smooth.argmax()] == x[-1]),
    saturation_at_upper_boundary=bool(s99 == x[-1]),
    near_best_later_outside_pct=float(100 * np.mean((y / x)[x >= s_near] > 1.05 * (y / x).min())),
    bootstrap_90pct=boot.quantile([.05, .95]).to_dict(),
    median_abs_tail_count_change_pct=float(tail_changes.tail_count_change_pct.abs().median()),
    models=parameters, comparison=fits.astype(object).where(pd.notna(fits), None).to_dict(orient="records"),
    scope="Finite measured grid; native 6-worker throughput-derived latency; retrospective fixed candidates; no automatic plateau claim",
)
# Cache reuse must not rewrite measurements or replace their hardware metadata.
assert chunk_latency.input_hashes(RUN) == source_hashes
(OUT / "findings.json").write_text(json.dumps(summary, indent=2, allow_nan=False))
display(Markdown(f"**Four measured/fitted lengths:** s99={s99:g}, raw argmin={s_argmin:g}, "
                 f"hinge={s_hinge:g}, constrained Ts={s_ts:g} KiB.\n\n"
                 "The first crossing, the efficiency minimum, and the two model-specific optima answer different questions."))

## Reuse and provenance

`runs/02_chunk_latency/{hardware.json,block_estimates.csv,io_raw.csv}` are read-only inputs during normal execution. Their hashes and acquisition metadata are copied into `breakpoint_comparison/input_provenance.json`. The prior top-level figures/findings are historical; the tables and plots produced by this revised notebook live under **`breakpoint_comparison/`**. No other notebook's saved model is silently replaced.

With `RERUN_MEASUREMENTS=True`, the original measurement protocol runs on the target storage device and returns a new `02_chunk_latency_measurement_...` directory. To reuse that new measurement in a later session, set `DATA_DIR` to that printed directory and set the switch back to `False`. An interrupted acquisition keeps `.incomplete` and cannot be mistaken for a completed cache.

A different device, reader configuration, row width, or workload needs its own validation. Predictions beyond the measured range and claims about real mask wall-clock latency are outside this experiment.